# 📘 03 — PyTorch Autograd: Computing Derivatives for PINNs

> **Goal:** Understand how PyTorch's automatic differentiation (autograd) engine works, and learn to compute first and higher-order derivatives — the core engine behind every PINN.

---
## 1. Why Autograd Matters for PINNs

In traditional numerical methods, derivatives are approximated using **finite differences**:

$$u'(t) \approx \frac{u(t + h) - u(t)}{h}$$

This is inaccurate (truncation error) and requires a grid.

PINNs take a completely different approach. The neural network $u_\theta(t)$ is a **differentiable function**, so we can compute its derivatives **exactly** using the chain rule — automatically.

This is called **Automatic Differentiation (AD)**, and PyTorch implements it via its `autograd` engine.

| Method | How derivatives are computed | Accuracy | Grid needed? |
|---|---|---|---|
| Finite Difference | $\frac{f(x+h)-f(x)}{h}$ | Approximate ($O(h)$) | ✅ Yes |
| Symbolic Diff | Manual algebra (e.g. SymPy) | Exact | ❌ No |
| Automatic Diff (AD) | Chain rule on computation graph | **Machine-precision exact** | ❌ No |

> 🔑 **Autograd = the engine that makes PINNs possible.** Without it, we cannot compute $\frac{\partial u_\theta}{\partial t}$ or $\frac{\partial^2 u_\theta}{\partial x^2}$ to enforce the PDE.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12
})

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## 2. The Computational Graph

Every time you perform operations on tensors in PyTorch, it secretly builds a **directed acyclic graph (DAG)** that records every operation.

```
Forward pass:
  t ──► (×2) ──► (cos) ──► u
              
Backward pass (autograd):
  du/dt ◄── (chain rule) ◄── stored graph
```

- **Leaf nodes:** Input tensors (e.g., `t`)
- **Non-leaf nodes:** Results of operations (e.g., `u = cos(2t)`)
- **Edges:** Record which operation created each tensor

When you call `.backward()` or `torch.autograd.grad()`, PyTorch **traverses this graph in reverse** and applies the chain rule at each node.

In [ ]:
# Visualizing the computation graph manually
t = torch.tensor(1.0, requires_grad=True)

# Forward pass: u = cos(2t)
a = 2 * t          # a = 2t
u = torch.cos(a)   # u = cos(2t)

print("=== Computational Graph Info ===")
print(f"t:   value={t.item():.4f}, requires_grad={t.requires_grad}, grad_fn={t.grad_fn}")
print(f"a=2t: value={a.item():.4f}, requires_grad={a.requires_grad}, grad_fn={a.grad_fn}")
print(f"u=cos(2t): value={u.item():.4f}, requires_grad={u.requires_grad}, grad_fn={u.grad_fn}")

print("\n=== Chain Rule Manually ===")
print("du/dt = du/da * da/dt = -sin(2t) * 2")
manual_grad = -2 * torch.sin(2 * t)
print(f"Manual: du/dt at t=1 = {manual_grad.item():.6f}")

# Let autograd compute it
du_dt = torch.autograd.grad(u, t)[0]
print(f"Autograd: du/dt at t=1 = {du_dt.item():.6f}")

---
## 3. Tensors & `requires_grad`

The key that enables gradient tracking is `requires_grad=True`.

| Setting | Meaning |
|---|---|
| `requires_grad=True` | Track all operations on this tensor |
| `requires_grad=False` | Default — no gradient tracking |
| `.detach()` | Stop gradient tracking for a tensor |
| `torch.no_grad()` | Context manager — disable tracking entirely |

In [ ]:
# --- Basic tensor gradient tracking ---
print("=== requires_grad examples ===")

# Default: no tracking
t1 = torch.tensor(2.0)
print(f"Default tensor: requires_grad = {t1.requires_grad}")

# Enabled tracking
t2 = torch.tensor(2.0, requires_grad=True)
print(f"With requires_grad=True: requires_grad = {t2.requires_grad}")

# Detach stops tracking
t3 = t2.detach()
print(f"After .detach(): requires_grad = {t3.requires_grad}")

print("\n=== PINN-style input tensor ===")
# This is how PINNs create collocation points
N = 100  # number of collocation points
t_colloc = torch.linspace(0, 1, N).reshape(-1, 1).requires_grad_(True)
print(f"Collocation tensor shape: {t_colloc.shape}")
print(f"requires_grad: {t_colloc.requires_grad}")
print(f"First 5 values: {t_colloc[:5].detach().squeeze().numpy()}")

---
## 4. `.backward()` — Backpropagation Basics

`.backward()` is the classic way to compute gradients — used in neural network training.

**It computes:** $\frac{\partial L}{\partial \theta}$ for all parameters $\theta$ in the network.

### Important rules:
- Only works on **scalar** outputs (or you must provide a `gradient` argument)
- Gradients **accumulate** in `.grad` — call `.zero_grad()` before each step
- The graph is **consumed** after `.backward()` unless `retain_graph=True`

In [ ]:
print("=== .backward() basics ===")

# Simple scalar case
t = torch.tensor(np.pi / 4, requires_grad=True, dtype=torch.float32)
u = torch.sin(t)   # u = sin(t), du/dt = cos(t)

u.backward()  # compute gradients

print(f"t = π/4 = {t.item():.6f}")
print(f"u = sin(t) = {u.item():.6f}")
print(f"du/dt via .grad = {t.grad.item():.6f}")
print(f"Expected cos(π/4) = {np.cos(np.pi/4):.6f}")

print("\n=== Gradient accumulation warning ===")
t = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    u = t ** 2
    u.backward()
    print(f"Step {i+1}: t.grad = {t.grad.item():.2f}  ← accumulates if not zeroed!")

print("\nSolution: zero out grad before each backward call")
t = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    if t.grad is not None:
        t.grad.zero_()
    u = t ** 2
    u.backward()
    print(f"Step {i+1}: t.grad = {t.grad.item():.2f}  ← correct!")

---
## 5. `torch.autograd.grad` — Explicit Derivative Computation

For PINNs, `.backward()` is **not enough** — we need derivatives of the network output with respect to its **inputs** (not parameters), and we need to do this **at arbitrary points**.

That's where `torch.autograd.grad` comes in.

```python
torch.autograd.grad(
    outputs,           # what to differentiate
    inputs,            # with respect to what
    grad_outputs,      # for non-scalar outputs (usually ones)
    create_graph,      # True = keep graph for higher-order derivatives
    retain_graph,      # True = don't destroy graph after
)
```

### `.backward()` vs `torch.autograd.grad`:
| | `.backward()` | `torch.autograd.grad()` |
|---|---|---|
| Output stored in | `.grad` attribute | Returned directly |
| Use case | Training loop (weight gradients) | PDE residuals (input gradients) |
| Supports `create_graph` | ✅ Yes | ✅ Yes |
| Non-scalar outputs | Need `gradient=` arg | Need `grad_outputs=` arg |
| More explicit control | ❌ No | ✅ Yes |

In [ ]:
print("=== torch.autograd.grad basics ===")

# Scalar input, scalar output
t = torch.tensor(1.0, requires_grad=True)
u = torch.sin(t)

du_dt = torch.autograd.grad(u, t)[0]
print(f"d/dt[sin(t)] at t=1: {du_dt.item():.6f}")
print(f"Expected cos(1):     {np.cos(1):.6f}")

print("\n=== Batch of inputs (PINN-style) ===")
# This is what we do inside a PINN training loop
t_batch = torch.linspace(0, 2*np.pi, 6, requires_grad=True).reshape(-1, 1)
u_batch = torch.sin(t_batch)

# grad_outputs=torch.ones_like(u_batch) means: sum over all outputs first
du_dt_batch = torch.autograd.grad(
    outputs=u_batch,
    inputs=t_batch,
    grad_outputs=torch.ones_like(u_batch),
    create_graph=True   # needed if we want to differentiate again
)[0]

print("t values:       ", t_batch.detach().squeeze().numpy().round(4))
print("u = sin(t):     ", u_batch.detach().squeeze().numpy().round(4))
print("du/dt = cos(t): ", du_dt_batch.detach().squeeze().numpy().round(4))
print("Expected cos(t):", np.cos(t_batch.detach().numpy().squeeze()).round(4))

---
## 6. First-Order Derivatives

Let's verify autograd against known analytical derivatives for several functions.

In [ ]:
def compute_derivative(func, t_vals):
    """Helper: compute df/dt for a given function over t_vals."""
    t = t_vals.clone().requires_grad_(True)
    f = func(t)
    df_dt = torch.autograd.grad(
        f, t,
        grad_outputs=torch.ones_like(f),
        create_graph=True
    )[0]
    return f.detach(), df_dt.detach()


t_vals = torch.linspace(0, 2*np.pi, 300).reshape(-1, 1)

# Define test functions and their known analytical derivatives
test_cases = [
    ("sin(t)",    lambda t: torch.sin(t),       lambda t: np.cos(t)),
    ("t³",        lambda t: t**3,               lambda t: 3*t**2),
    ("exp(-t)",   lambda t: torch.exp(-t),      lambda t: -np.exp(-t)),
    ("tanh(t)",   lambda t: torch.tanh(t),      lambda t: 1 - np.tanh(t)**2),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()
t_np = t_vals.numpy().squeeze()

for ax, (name, func_torch, func_analytical) in zip(axes, test_cases):
    f, df = compute_derivative(func_torch, t_vals)
    df_exact = func_analytical(t_np)

    ax.plot(t_np, f.numpy().squeeze(), 'royalblue', lw=2, label=f'$f(t)$ = {name}')
    ax.plot(t_np, df.numpy().squeeze(), 'tomato', lw=2, label="Autograd $f'(t)$")
    ax.plot(t_np, df_exact, 'k--', lw=1.5, alpha=0.6, label="Analytical $f'(t)$")

    max_err = np.max(np.abs(df.numpy().squeeze() - df_exact))
    ax.set_title(f"$f(t) = $ {name}  |  Max error: {max_err:.2e}")
    ax.legend(fontsize=9)
    ax.set_xlabel('$t$')

plt.suptitle('First-Order Derivatives: Autograd vs Analytical', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Higher-Order Derivatives (for PDE Terms)

Many PDEs require second or even higher-order derivatives:

- Heat equation: $\dfrac{\partial^2 u}{\partial x^2}$
- Wave equation: $\dfrac{\partial^2 u}{\partial t^2}$
- Beam bending: $\dfrac{\partial^4 u}{\partial x^4}$

**Key requirement:** Use `create_graph=True` when computing the first derivative — this keeps the graph alive so we can differentiate again.

In [ ]:
def nth_derivative(func, t_vals, n=1):
    """
    Compute the nth derivative of func w.r.t. t_vals using autograd.
    Requires create_graph=True for all intermediate steps.
    """
    t = t_vals.clone().requires_grad_(True)
    u = func(t)
    for _ in range(n):
        u = torch.autograd.grad(
            u, t,
            grad_outputs=torch.ones_like(u),
            create_graph=True,   # ← CRITICAL for higher-order derivatives
            retain_graph=True
        )[0]
    return u.detach()


t_vals = torch.linspace(-np.pi, np.pi, 400).reshape(-1, 1)
t_np   = t_vals.numpy().squeeze()

# --- sin(t) and its derivatives ---
# sin(t) → cos(t) → -sin(t) → -cos(t) → sin(t) ...
u   = nth_derivative(torch.sin, t_vals, n=0)  # sin(t)
u1  = nth_derivative(torch.sin, t_vals, n=1)  # cos(t)
u2  = nth_derivative(torch.sin, t_vals, n=2)  # -sin(t)
u3  = nth_derivative(torch.sin, t_vals, n=3)  # -cos(t)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t_np, u.squeeze(),  lw=2.5, label='$u = \\sin(t)$',     color='royalblue')
ax.plot(t_np, u1.squeeze(), lw=2.5, label="$u' = \\cos(t)$",    color='green')
ax.plot(t_np, u2.squeeze(), lw=2.5, label="$u'' = -\\sin(t)$",  color='tomato')
ax.plot(t_np, u3.squeeze(), lw=2.5, label="$u''' = -\\cos(t)$", color='purple')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('$t$')
ax.set_title('Higher-Order Derivatives of sin(t) via Autograd', fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Verify with analytical values
max_err_d2 = np.max(np.abs(u2.numpy().squeeze() - (-np.sin(t_np))))
max_err_d3 = np.max(np.abs(u3.numpy().squeeze() - (-np.cos(t_np))))
print(f"Max error in d²/dt² [sin(t)]:  {max_err_d2:.2e}")
print(f"Max error in d³/dt³ [sin(t)]:  {max_err_d3:.2e}")

In [ ]:
print("=== Partial Derivatives (2D input) ===")
print("This is what we need for PDEs like: ∂²u/∂x² + ∂²u/∂t²")

# u(x, t) = sin(πx) * cos(πt)  — solution to wave equation
# ∂u/∂x = π*cos(πx)*cos(πt)
# ∂u/∂t = -π*sin(πx)*sin(πt)
# ∂²u/∂x² = -π²*sin(πx)*cos(πt)
# ∂²u/∂t² = -π²*sin(πx)*cos(πt)  →  ∂²u/∂t² = ∂²u/∂x²  (wave equation ✓)

x = torch.tensor([[0.25], [0.5], [0.75]], requires_grad=True, dtype=torch.float32)
t = torch.tensor([[0.1],  [0.2], [0.3]],  requires_grad=True, dtype=torch.float32)

u = torch.sin(np.pi * x) * torch.cos(np.pi * t)

# ∂u/∂x
du_dx = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
# ∂u/∂t
du_dt = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
# ∂²u/∂x²
d2u_dx2 = torch.autograd.grad(du_dx, x, grad_outputs=torch.ones_like(du_dx), create_graph=True)[0]
# ∂²u/∂t²
d2u_dt2 = torch.autograd.grad(du_dt, t, grad_outputs=torch.ones_like(du_dt), create_graph=True)[0]

print("\n  x      t      u         ∂u/∂x     ∂u/∂t     ∂²u/∂x²   ∂²u/∂t²")
print("-" * 75)
for i in range(3):
    xi = x[i].item(); ti = t[i].item()
    print(f"  {xi:.2f}   {ti:.2f}   "
          f"{u[i].item():+.4f}   "
          f"{du_dx[i].item():+.4f}   "
          f"{du_dt[i].item():+.4f}   "
          f"{d2u_dx2[i].item():+.4f}   "
          f"{d2u_dt2[i].item():+.4f}")

print("\nWave equation check: ∂²u/∂t² - ∂²u/∂x² should be ≈ 0")
residual = d2u_dt2 - d2u_dx2
print("Residuals:", residual.detach().numpy().squeeze().round(6))

---
## 8. Derivatives Through a Neural Network

This is the **heart of PINNs**. The neural network approximates the solution $u_\theta(t)$, and autograd lets us compute $\frac{du_\theta}{dt}$, $\frac{d^2u_\theta}{dt^2}$, etc., exactly.

In [ ]:
class SimpleNet(nn.Module):
    """
    A simple fully-connected network.
    Input:  t ∈ ℝ  (scalar time)
    Output: u ∈ ℝ  (approximated solution)
    """
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, t):
        return self.net(t)


def get_derivatives(model, t_in):
    """
    Compute u, du/dt, d²u/dt² from the model output.
    t_in must have requires_grad=True.
    """
    u = model(t_in)

    du_dt = torch.autograd.grad(
        u, t_in,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    d2u_dt2 = torch.autograd.grad(
        du_dt, t_in,
        grad_outputs=torch.ones_like(du_dt),
        create_graph=True
    )[0]

    return u, du_dt, d2u_dt2


torch.manual_seed(42)
model = SimpleNet(hidden=32)

t_test = torch.linspace(0, 2*np.pi, 200).reshape(-1, 1).requires_grad_(True)
u, du, d2u = get_derivatives(model, t_test)

t_np = t_test.detach().numpy().squeeze()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(t_np, u.detach().numpy().squeeze(), 'royalblue', lw=2)
axes[0].set_title('Network Output $u_\\theta(t)$')
axes[0].set_xlabel('$t$')

axes[1].plot(t_np, du.detach().numpy().squeeze(), 'green', lw=2)
axes[1].set_title("First Derivative $du_\\theta/dt$")
axes[1].set_xlabel('$t$')

axes[2].plot(t_np, d2u.detach().numpy().squeeze(), 'tomato', lw=2)
axes[2].set_title("Second Derivative $d^2u_\\theta/dt^2$")
axes[2].set_xlabel('$t$')

plt.suptitle('Autograd Derivatives Through a Neural Network (untrained)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Network parameter count: {sum(p.numel() for p in model.parameters()):,}")
print("Note: These are random (untrained) — they will match the target after PINN training!")

---
## 9. Computing PDE Residuals with Autograd

Now let's put everything together and compute actual **ODE/PDE residuals** — exactly what a PINN loss function does.

### 9.1 ODE Residual: Exponential Decay
$$\frac{du}{dt} + k\,u = 0$$

In [ ]:
def ode_residual_exponential_decay(model, t, k=1.0):
    """
    Physics residual for: du/dt + k*u = 0
    A perfect solution gives residual = 0 everywhere.
    """
    u = model(t)
    du_dt = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    return du_dt + k * u


# --- Untrained network: large residual ---
torch.manual_seed(0)
model = SimpleNet(hidden=16)
t_colloc = torch.linspace(0, 5, 100).reshape(-1, 1).requires_grad_(True)

res_untrained = ode_residual_exponential_decay(model, t_colloc, k=1.0)
print("=== ODE Residual Check ===")
print(f"Untrained network — Mean |residual|: {res_untrained.abs().mean().item():.4f}")

# --- "Perfect" solution: u(t) = e^(-t), wrapped as a net-like callable ---
class ExactSolution(nn.Module):
    """Wraps the analytical solution u = e^(-t) as a callable module."""
    def forward(self, t):
        return torch.exp(-t)

exact_model = ExactSolution()
res_exact = ode_residual_exponential_decay(exact_model, t_colloc, k=1.0)
print(f"Exact solution     — Mean |residual|: {res_exact.abs().mean().item():.2e}  (≈ 0 ✓)")

### 9.2 ODE Residual: Simple Harmonic Oscillator
$$u'' + \omega^2\, u = 0$$

In [ ]:
def ode_residual_sho(model, t, omega=2.0):
    """
    Physics residual for: u'' + omega^2 * u = 0
    """
    u = model(t)
    du_dt = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    d2u_dt2 = torch.autograd.grad(
        du_dt, t,
        grad_outputs=torch.ones_like(du_dt),
        create_graph=True
    )[0]
    return d2u_dt2 + omega**2 * u


omega = 2.0

class SHOExact(nn.Module):
    """Exact: u(t) = cos(omega * t)"""
    def forward(self, t):
        return torch.cos(omega * t)

t_test = torch.linspace(0, 4*np.pi/omega, 200).reshape(-1, 1).requires_grad_(True)

res_sho_exact    = ode_residual_sho(SHOExact(), t_test, omega=omega)
res_sho_untrained = ode_residual_sho(SimpleNet(32), t_test, omega=omega)

print("=== SHO Residual ===")
print(f"Exact solution     — Mean |residual|: {res_sho_exact.abs().mean().item():.2e}  (≈ 0 ✓)")
print(f"Untrained network  — Mean |residual|: {res_sho_untrained.abs().mean().item():.4f}")

# Visualize residuals
t_np = t_test.detach().numpy().squeeze()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(t_np, res_sho_exact.detach().numpy().squeeze(), 'green', lw=2)
axes[0].set_title('SHO Residual: Exact Solution (≈ 0)')
axes[0].set_xlabel('$t$')
axes[0].set_ylabel('Residual')

axes[1].plot(t_np, res_sho_untrained.detach().numpy().squeeze(), 'tomato', lw=2)
axes[1].set_title('SHO Residual: Untrained Network (large)')
axes[1].set_xlabel('$t$')
axes[1].set_ylabel('Residual')

plt.suptitle(r"$u'' + \omega^2 u = 0$ — Physics Residual", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.3 Full PINN Loss Function (Preview)

Here's the complete structure of a PINN loss — this is what you'll implement in the next notebooks.

In [ ]:
def pinn_loss(model, t_colloc, t_ic, u_ic, omega=2.0):
    """
    Full PINN loss for the Simple Harmonic Oscillator:
        u'' + omega^2 * u = 0
        u(0) = 1,  u'(0) = 0

    Loss = λ_phys * L_physics + λ_ic * L_ic
    """
    # --- 1. Physics Residual Loss ---
    u = model(t_colloc)
    du_dt = torch.autograd.grad(
        u, t_colloc,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    d2u_dt2 = torch.autograd.grad(
        du_dt, t_colloc,
        grad_outputs=torch.ones_like(du_dt),
        create_graph=True
    )[0]

    residual = d2u_dt2 + omega**2 * u
    loss_physics = torch.mean(residual**2)

    # --- 2. Initial Condition Loss ---
    u_pred_ic = model(t_ic)                # u(0) prediction
    du_dt_ic  = torch.autograd.grad(
        u_pred_ic, t_ic,
        grad_outputs=torch.ones_like(u_pred_ic),
        create_graph=True
    )[0]                                   # u'(0) prediction

    loss_ic_u  = (u_pred_ic  - u_ic[0])**2   # u(0) = 1
    loss_ic_du = (du_dt_ic   - u_ic[1])**2   # u'(0) = 0
    loss_ic    = loss_ic_u + loss_ic_du

    # --- Total Loss ---
    total = loss_physics + loss_ic
    return total, loss_physics.item(), loss_ic.item()


# Test the loss function on an untrained vs exact model
torch.manual_seed(42)
model      = SimpleNet(hidden=32)
exact_model = SHOExact()
omega = 2.0

t_colloc = torch.linspace(0, 4*np.pi/omega, 200).reshape(-1, 1).requires_grad_(True)
t_ic     = torch.tensor([[0.0]], requires_grad=True)
u_ic     = [torch.tensor([[1.0]]), torch.tensor([[0.0]])]  # [u(0), u'(0)]

total_u,  phys_u,  ic_u  = pinn_loss(model, t_colloc, t_ic, u_ic, omega)
total_e,  phys_e,  ic_e  = pinn_loss(exact_model, t_colloc, t_ic, u_ic, omega)

print("=" * 50)
print("          PINN Loss Breakdown")
print("=" * 50)
print(f"{'':20s}  {'Physics':>10s}  {'IC':>10s}  {'Total':>10s}")
print("-" * 50)
print(f"{'Untrained Network':20s}  {phys_u:>10.4f}  {ic_u:>10.4f}  {total_u.item():>10.4f}")
print(f"{'Exact Solution':20s}  {phys_e:>10.2e}  {ic_e:>10.2e}  {total_e.item():>10.2e}")
print("=" * 50)
print("\nGoal of PINN training: minimize all losses → both → ≈ 0")

---
## 10. Common Mistakes & How to Avoid Them

In [ ]:
print("=" * 55)
print("  COMMON AUTOGRAD MISTAKES IN PINNs")
print("=" * 55)

# -------------------------------------------------------
# MISTAKE 1: Forgetting requires_grad=True on inputs
# -------------------------------------------------------
print("\n❌ Mistake 1: No requires_grad on input")
t_bad = torch.linspace(0, 1, 10).reshape(-1, 1)  # no requires_grad!
u_bad = torch.sin(t_bad)
try:
    grad = torch.autograd.grad(u_bad, t_bad, grad_outputs=torch.ones_like(u_bad))[0]
except RuntimeError as e:
    print(f"   Error: {str(e)[:80]}")

print("✅ Fix: add requires_grad_(True)")
t_good = torch.linspace(0, 1, 10).reshape(-1, 1).requires_grad_(True)
u_good = torch.sin(t_good)
grad = torch.autograd.grad(u_good, t_good, grad_outputs=torch.ones_like(u_good))[0]
print(f"   Gradient computed successfully, shape: {grad.shape}")

# -------------------------------------------------------
# MISTAKE 2: Forgetting create_graph=True for higher-order
# -------------------------------------------------------
print("\n❌ Mistake 2: No create_graph for 2nd derivative")
t = torch.linspace(0, 1, 10).reshape(-1, 1).requires_grad_(True)
u = torch.sin(t)
du_dt = torch.autograd.grad(
    u, t,
    grad_outputs=torch.ones_like(u),
    create_graph=False  # ← wrong!
)[0]
try:
    d2u_dt2 = torch.autograd.grad(du_dt, t, grad_outputs=torch.ones_like(du_dt))[0]
except RuntimeError as e:
    print(f"   Error: {str(e)[:80]}")

print("✅ Fix: use create_graph=True in every intermediate grad call")
t = torch.linspace(0, 1, 10).reshape(-1, 1).requires_grad_(True)
u = torch.sin(t)
du_dt = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
d2u_dt2 = torch.autograd.grad(du_dt, t, grad_outputs=torch.ones_like(du_dt), create_graph=True)[0]
print(f"   d²u/dt² computed, shape: {d2u_dt2.shape}")

# -------------------------------------------------------
# MISTAKE 3: Using .numpy() before .detach()
# -------------------------------------------------------
print("\n❌ Mistake 3: Calling .numpy() on tensor that requires grad")
t = torch.tensor([1.0, 2.0], requires_grad=True)
try:
    arr = t.numpy()
except RuntimeError as e:
    print(f"   Error: {str(e)[:80]}")

print("✅ Fix: always call .detach() before .numpy()")
arr = t.detach().numpy()
print(f"   Converted safely: {arr}")

# -------------------------------------------------------
# MISTAKE 4: Gradient accumulation in training loops
# -------------------------------------------------------
print("\n❌ Mistake 4: Forgetting optimizer.zero_grad() before loss.backward()")
print("   → Gradients accumulate across steps → wrong parameter updates")
print("✅ Fix: always call optimizer.zero_grad() at the start of each step:")
print("""
   for epoch in range(epochs):
       optimizer.zero_grad()   # ← clear accumulated grads
       loss = pinn_loss(...)   # forward pass
       loss.backward()         # compute gradients
       optimizer.step()        # update weights
""")

print("=" * 55)

In [ ]:
# Quick reference card
print("""
╔══════════════════════════════════════════════════════════════╗
║           AUTOGRAD QUICK REFERENCE FOR PINNs                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Create input:                                               ║
║    t = torch.linspace(0,1,N).reshape(-1,1).requires_grad_(True) ║
║                                                              ║
║  First derivative:                                           ║
║    du = torch.autograd.grad(                                 ║
║        u, t,                                                 ║
║        grad_outputs=torch.ones_like(u),                      ║
║        create_graph=True   ← always True for PINNs           ║
║    )[0]                                                      ║
║                                                              ║
║  Second derivative:                                          ║
║    d2u = torch.autograd.grad(                                ║
║        du, t,                                                ║
║        grad_outputs=torch.ones_like(du),                     ║
║        create_graph=True                                     ║
║    )[0]                                                      ║
║                                                              ║
║  Physics residual (SHO):                                     ║
║    residual = d2u + omega**2 * u                             ║
║    loss_phys = torch.mean(residual**2)                       ║
║                                                              ║
║  For plotting: u.detach().numpy()                            ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 11. Summary & Key Takeaways

### ✅ What we covered:

| Topic | Key Point |
|---|---|
| Why autograd | Computes exact derivatives at any point — no grid required |
| Computational graph | DAG built during forward pass; traversed in reverse for gradients |
| `requires_grad=True` | Must be set on inputs you want to differentiate w.r.t. |
| `.backward()` | For computing weight gradients in the training loop |
| `torch.autograd.grad` | For computing input derivatives (PDE terms) explicitly |
| `create_graph=True` | Required for higher-order derivatives (2nd, 3rd, ...) |
| Partial derivatives | Use separate `requires_grad` inputs for $x$ and $t$ |
| PDE residuals | Autograd computes terms like $u_{tt}$, $u_{xx}$ — the physics loss |
| Common mistakes | `requires_grad`, `create_graph`, gradient accumulation, `.detach()` |

### 🔜 Next Steps:

| Notebook | What You'll Learn |
|---|---|
| `02_pinn_theory/01_what_is_pinn.ipynb` | Full PINN architecture and training loop |
| `03_pinn_basics/01_simple_ode.ipynb` | Train a PINN to solve the harmonic oscillator |
| `03_pinn_basics/02_heat_equation_1d.ipynb` | Extend to PDEs with spatial derivatives |

### 📖 References:
- [PyTorch Autograd documentation](https://pytorch.org/docs/stable/autograd.html)
- [PyTorch Autograd tutorial](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)
- Baydin et al. (2018) — *Automatic Differentiation in Machine Learning: a Survey*
- Raissi et al. (2019) — *Physics-Informed Neural Networks*